##How to access using  Service Principal

In [0]:
service_credential = dbutils.secrets.get(scope='fabchi_scope', key='app-secret')

spark.conf.set("fs.azure.account.auth.type.fabchistorage001.dfs.core.windows.net", "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type.fabchistorage001.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id.fabchistorage001.dfs.core.windows.net", "616b5348-9dba-4607-bcec-baa1c23c4d03")
spark.conf.set("fs.azure.account.oauth2.client.secret.fabchistorage001.dfs.core.windows.net", service_credential)
spark.conf.set("fs.azure.account.oauth2.client.endpoint.fabchistorage001.dfs.core.windows.net", "https://login.microsoftonline.com/75058155-f8e7-4912-bb8a-9e774be50420/oauth2/token")

### **dbutils.secrets**

In [0]:
dbutils.secrets.listScopes()

In [0]:
dbutils.secrets.list(scope='fabchi_scope')

In [0]:
dbutils.secrets.get(scope='fabchi_scope', key='app-secret')

In [0]:
dbutils.fs.ls('/')

## **Reading Data** 

In [0]:
dbutils.fs.ls("abfss://salescontainer@fabchistorage001.dfs.core.windows.net/")

In [0]:
sales_df = spark.read.format('csv')\
    .option('header', 'true')\
    .option('inferSchema', 'true')\
    .load("abfss://salescontainer@fabchistorage001.dfs.core.windows.net/26-03-27/sales.csv")
display(sales_df)


### Data Transformation with Pyspark

In [0]:
sales_df.describe()
sales_df.printSchema()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
sales_df.withColumn('Item_Type', split(col('Item_Type'),' ')).display()

In [0]:
# Counting null values in a single column

sales_df.select(count(when(isnull('Item_Weight'), 'Item_Weight')).alias("null_count")).show()

In [0]:
# Counting of nulls in Multiple coulmns

null_counts = sales_df.select([count(when(isnull(c), c)).alias(c) for c in sales_df.columns])


In [0]:
null_counts.display()

In [0]:
# Creating a new column with a new datatype
sales_df.withColumn('Item_Visibility',col('Item_Visibility').cast(StringType())).display()

In [0]:
# counting duplicates through a particular column
sales_df.groupBy('Item_Identifier').count().filter(col('count') > 1).display()





In [0]:
%sql
drop catalog if exists moviecatalog cascade

In [0]:
%sql
-- Creating of catalog

CREATE SCHEMA IF NOT EXISTS moviecatalog.staging;

drop schema moviec



In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS moviecatalog.;

In [0]:
bronze_path = 'abfss://moviedbcontainer@fabchistorage001.dfs.core.windows.net/rawmdb/movie.csv'

df = spark.read.format('csv')\
    .option('header', 'True')\
    .option('inferSchema', 'True')\
    .load(bronze_path)

In [0]:
null_counts = df.select([count(when(isnull(c), c)).alias(c) for c in df.columns])

null_counts.display()